# 01. VGGFace2 목록 준비
같은 목록과 label mapping을 모든 seed/head가 공유합니다.

## 1. 설정

In [1]:
from pathlib import Path
import sys, json
BASE_DIR = Path("F:/FaceX-Zoo")
SETTINGS_PATH = BASE_DIR / "fine-tune/data/notebook_settings.json"
for path in (BASE_DIR, BASE_DIR / "fine-tune"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
from modules.notebook_workflow import resolve_settings, save_settings, plan_matrix
# 00의 공통값을 읽습니다. 아래 변수 변경이 저장값/YAML보다 우선합니다.
saved = resolve_settings(BASE_DIR, SETTINGS_PATH, require_saved=True)
SEEDS = list(saved["project"]["seed_list"])  # 예: [8972]. 이후 단계도 같은 목록을 사용하세요.
HEADS = list(saved["campaign"]["heads"])  # 기본 세 head. 02는 ArcFace 부모만 생성합니다.
KEEP_RAW_RESULTS = saved["project"]["keep_raw_results"]
NUM_WORKERS = 0
DEVICE = saved["project"]["device"]
DATASET = dict(saved["dataset"])  # 경로·input_size·pixel_mean/std
BACKBONE = dict(saved["backbone"])
TRAINING = dict(saved["training"])  # epoch·batch·학습률 등
HEAD_PARAMETERS = saved["heads"]
PATHS = dict(saved["paths"])
REUSE_COMPLETED = saved["campaign"]["reuse_completed"]
EXECUTE = True  # True일 때만 목록 생성/학습/export를 수행합니다.
EXTRACT_TRAIN_ARCHIVE = False  # 로컬 train 압축파일 해제. EXECUTE=True일 때만 실행
VAL_EVERY = 20  # 매 20번째 이미지를 validation으로 분리 (자체 선택)
# 현재 노트북 변수 -> 실제 모듈 설정. 입력 112x112 이외는 현 구조에서 명시적 중단.
DATASET.update(train_archive=str(Path(DATASET["raw_root"])/"data/vggface2_train.tar.gz"),
               test_archive=str(Path(DATASET["raw_root"])/"data/vggface2_test.tar.gz"),
               meta_csv=str(Path(DATASET["raw_root"])/"meta/identity_meta.csv"))
settings = resolve_settings(BASE_DIR, SETTINGS_PATH, overrides={
    "project": {"seed": SEEDS[0], "seed_list": SEEDS, "keep_raw_results": KEEP_RAW_RESULTS, "device": DEVICE},
    "campaign": {"heads": HEADS, "reuse_completed": REUSE_COMPLETED},
    "dataset": DATASET, "backbone": BACKBONE, "heads": HEAD_PARAMETERS,
    "training": {**TRAINING, "num_workers": NUM_WORKERS}, "paths": PATHS,
})


## 2. 경로 확인
압축 해제는 원본 JPEG를 `DATASET["raw_root"]/train/` 아래에 저장합니다. 얼굴 정렬이나 리사이즈를 수행하는 단계는 아닙니다. 로컬 압축파일을 사용하며 다운로드하지 않습니다.

In [2]:
print(DATASET)
print("EXECUTE:", EXECUTE, "VAL_EVERY:", VAL_EVERY)
print("EXTRACT_TRAIN_ARCHIVE:", EXTRACT_TRAIN_ARCHIVE)
print("extract:", DATASET["train_archive"], "->", DATASET["raw_root"])

{'name': 'VGGFace2', 'raw_root': 'F:/FaceX-Zoo/VGGFace2', 'image_root': 'F:/FaceX-Zoo/VGGFace2/train', 'train_archive': 'F:\\FaceX-Zoo\\VGGFace2\\data\\vggface2_train.tar.gz', 'test_archive': 'F:\\FaceX-Zoo\\VGGFace2\\data\\vggface2_test.tar.gz', 'meta_csv': 'F:\\FaceX-Zoo\\VGGFace2\\meta\\identity_meta.csv', 'num_classes': 8631, 'input_size': [112, 112], 'pixel_mean': 127.5, 'pixel_std': 128.0, 'processed_root': 'F:/FaceX-Zoo/fine-tune/data/vggface2_processed', 'train_list': 'F:/FaceX-Zoo/fine-tune/data/train_list.txt', 'val_list': 'F:/FaceX-Zoo/fine-tune/data/val_list.txt'}
EXECUTE: True VAL_EVERY: 20
EXTRACT_TRAIN_ARCHIVE: False
extract: F:\FaceX-Zoo\VGGFace2\data\vggface2_train.tar.gz -> F:/FaceX-Zoo/VGGFace2


## 3. 학습 이미지 압축 해제 및 목록 생성
최초 실행은 전체 압축 해제에 시간이 걸립니다. 약 10초마다 처리한 파일 수를 표시합니다. 중단 후 다시 실행하면 압축파일을 처음부터 읽되, 기존 파일은 내용이 동일한지 확인하고 없는 파일만 추가합니다. 기존 파일 내용이 다르면 덮어쓰지 않고 중단합니다.

압축 해제를 이미 완료했다면 첫 셀에서 `EXTRACT_TRAIN_ARCHIVE=False`로 설정할 수 있습니다. 기존 train/val 목록은 그대로 검증·재사용합니다.

In [3]:
from modules.archive_preparation import extract_train_archive
from modules.notebook_workflow import prepare_indices
extraction_result = None
if EXECUTE:
    if EXTRACT_TRAIN_ARCHIVE:
        extraction_result = extract_train_archive(DATASET["train_archive"], DATASET["raw_root"])
    split_result = prepare_indices(settings, VAL_EVERY)
    print(split_result)
else:
    split_result = None
    print("계획 모드: 압축 해제 및 목록 생성 미실행")

{'train_count': 2984795, 'val_count': 157095, 'train_sha256': '3a26b7c45ccf1ff5125ae505e3e6a208367d983ffd7adc91f6da0973358ecad0', 'val_sha256': 'a98b2b4ac537240ba411f24b09696014b57b1c7c5cb8d9fdda04ad80ed1571ef', 'split': 'every 20th source row is validation; not identity-disjoint'}


## 4. 실제 이미지 전처리 확인

In [4]:
from modules.notebook_workflow import make_loaders
if EXECUTE:
    train_loader, val_loader = make_loaders(settings)
    images, labels = next(iter(val_loader))
    print(tuple(images.shape), float(images.min()), float(images.max()))
    print("workers:", train_loader.num_workers, val_loader.num_workers)
    del train_loader, val_loader
else:
    print("계획 모드: 이미지 읽기 미실행")

(64, 3, 112, 112) -0.99609375 0.99609375
workers: 0 0


## 5. 결과 해석

In [5]:
print("이미지 단위 분할이며 identity-disjoint 검증이 아닙니다.")
print("기존 목록과 다르면 덮어쓰지 않습니다. 새 목록 경로를 지정하세요.")
print("표본 존재/읽기 확인은 전체 데이터 무결성 확인이 아닙니다.")
print("압축 해제 완료는 전체 이미지 decode나 얼굴 정렬 검증 완료를 의미하지 않습니다.")

이미지 단위 분할이며 identity-disjoint 검증이 아닙니다.
기존 목록과 다르면 덮어쓰지 않습니다. 새 목록 경로를 지정하세요.
표본 존재/읽기 확인은 전체 데이터 무결성 확인이 아닙니다.
압축 해제 완료는 전체 이미지 decode나 얼굴 정렬 검증 완료를 의미하지 않습니다.
